# RAG

In [ ]:
import time
start_time = time.time()

### Upload html documents from local folder

BSHTMLLoader: Strips all HTML immediately → tables become unformatted text  
Your custom function: Converts tables to markdown first → tables remain structured

In [ ]:
# Upload all files in folder "6k_filings"
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
from langchain_core.documents import Document

def html_to_clean_text(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    
    tables = soup.find_all('table')
    for table in tables:
        try:
            df = pd.read_html(StringIO(str(table)))[0]
            markdown = "\n" + df.to_markdown(index=False) + "\n"
            table.replace_with(soup.new_string(markdown))
        except:
            table.replace_with(soup.new_string(table.get_text(separator=' ', strip=True)))
    
    return soup.get_text(separator='\n', strip=True)

# Load all files
folder_path = "./6k_filings"
htm_files = list(Path(folder_path).glob("*.htm"))

documents = []
for doc_id, file_path in enumerate(htm_files):
    html_content = file_path.read_text(encoding='utf-8')
    clean_text = html_to_clean_text(html_content)
    
    doc = Document(
        page_content=clean_text,
        metadata={
            "source": str(file_path),
            "docid": doc_id + 65456 
        }
    )
    documents.append(doc)

### Split LangChain document objects into chunks that are as well LangChain document objects

Considered using MarkdownHeaderTextSplitter because used markdowns to clarify tables. Still, better RecursiveCharacterTextSplitter

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,    #  adds where each chunk starts in the original document
    separators=["\n\n", "\n", ". ", " ", ""]  # Splits on paragraphs, then lines, then sentences, then words, then characters
)
chunks = text_splitter.split_documents(documents)
print(f'Split {len(documents)} filings (documents) into {len(chunks)} chunks.' )

Split 1010 filings (documents) into 63427 chunks.


### Batch Embeddings
#### Prepare batches

In [3]:
import json
from pathlib import Path

def create_batch_jsonl(
    chunks, 
    output_dir="batch_files",
    max_lines_per_file=10000
):
    """
    Create JSONL files for OpenAI batch embeddings with custom IDs and file limits
    
    Args:
        chunks: List of LangChain Document objects
        output_dir: Directory to save batch files
        max_lines_per_file: Maximum number of tasks per JSONL file
    
    Returns:
        List of created file paths
    """
    Path(output_dir).mkdir(exist_ok=True)
    batch_files = []
    
    # Split chunks into batches
    for batch_num in range(0, len(chunks), max_lines_per_file):
        batch_chunks = chunks[batch_num:batch_num + max_lines_per_file]
        output_file = f"{output_dir}/batch_for_embeddings_{batch_num // max_lines_per_file + 1}.jsonl"
        
        with open(output_file, 'w', encoding='utf-8') as f:
            for chunk in batch_chunks:
                # Create unique custom_id from metadata
                custom_id = (
                    str(chunk.metadata['docid']) + "_" + 
                    str(chunk.metadata['start_index'])
                )
                
                out_dict = {
                    "custom_id": custom_id,
                    "method": "POST",
                    "url": "/v1/embeddings",
                    "body": {
                        "model": "text-embedding-3-small",
                        "input": chunk.page_content
                    }
                }
                f.write(json.dumps(out_dict, ensure_ascii=False) + '\n')
        
        batch_files.append(output_file)
        print(f"Created {output_file} with {len(batch_chunks)} tasks")
    
    print(f"\nTotal: {len(batch_files)} batch file(s) created")
    return batch_files

# Create batch files
batch_files = create_batch_jsonl(chunks, max_lines_per_file=10000)

Created batch_files/batch_for_embeddings_1.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_2.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_3.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_4.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_5.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_6.jsonl with 10000 tasks
Created batch_files/batch_for_embeddings_7.jsonl with 3427 tasks

Total: 7 batch file(s) created


### Upload input file

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [2]:
from openai import OpenAI

client = OpenAI()
files = client.files.list()

In [8]:
len(files.to_dict()['data'])

375

In [9]:
from glob import glob

batch_files = glob('./batch_files/batch_for_embeddings_*.jsonl')
batch_files

['./batch_files\\batch_for_embeddings_1.jsonl',
 './batch_files\\batch_for_embeddings_2.jsonl',
 './batch_files\\batch_for_embeddings_3.jsonl',
 './batch_files\\batch_for_embeddings_4.jsonl']

In [10]:
from tqdm import tqdm
client = OpenAI()

my_batch_files_ids = []
for b_file in tqdm(batch_files):
    batch_input_file = client.files.create(
        file=open(b_file, "rb"), 
        purpose='batch'
    )
    my_batch_files_ids.append(batch_input_file.id)
    print(batch_input_file)

 25%|██▌       | 1/4 [00:08<00:26,  8.76s/it]

FileObject(id='file-B2c5GDq6YpD3FRsGYerqXY', bytes=9036446, created_at=1762847351, filename='batch_for_embeddings_1.jsonl', object='file', purpose='batch', status='processed', expires_at=1765439351, status_details=None)


 50%|█████     | 2/4 [00:17<00:17,  8.62s/it]

FileObject(id='file-TdCPrLozvZjUEzTRAJTNG2', bytes=9538759, created_at=1762847360, filename='batch_for_embeddings_2.jsonl', object='file', purpose='batch', status='processed', expires_at=1765439360, status_details=None)


 75%|███████▌  | 3/4 [00:36<00:13, 13.42s/it]

FileObject(id='file-HM7n2ZPgcYnRJmr1sEJ7DZ', bytes=9022641, created_at=1762847368, filename='batch_for_embeddings_3.jsonl', object='file', purpose='batch', status='processed', expires_at=1765439368, status_details=None)


100%|██████████| 4/4 [00:44<00:00, 11.08s/it]

FileObject(id='file-CNWRj2LBN7XrYdttoS5BSh', bytes=8706316, created_at=1762847386, filename='batch_for_embeddings_4.jsonl', object='file', purpose='batch', status='processed', expires_at=1765439386, status_details=None)


In [11]:
my_batch_files_ids

['file-B2c5GDq6YpD3FRsGYerqXY',
 'file-TdCPrLozvZjUEzTRAJTNG2',
 'file-HM7n2ZPgcYnRJmr1sEJ7DZ',
 'file-CNWRj2LBN7XrYdttoS5BSh']

In [12]:
my_id = 'antonio_m_lancuentra'

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
batch_description = f"Content embeddings ({my_id}) {timestamp}"

for file_id in tqdm(my_batch_files_ids):
    client.batches.create(
            input_file_id = file_id,
            endpoint="/v1/embeddings",
            completion_window="24h",
            metadata={
                "description": batch_description,
                "timestamp": timestamp
            }
        )

100%|██████████| 4/4 [00:02<00:00,  1.55it/s]


In [ ]:
batch_description

NameError: name 'my_id' is not defined

### After launching batches, I turn the computer off. Then I need to run the cells below

In [ ]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [ ]:
from openai import OpenAI

client = OpenAI()


In [5]:
client.files.list().to_dict()

{'data': [{'id': 'file-7Lg8rRExqXs1yGR8ertdZ5',
   'bytes': 318,
   'created_at': 1762849344,
   'filename': 'batch_6912eab1b75481909cbafc33a1401421_error.jsonl',
   'object': 'file',
   'purpose': 'batch_output',
   'status': 'processed',
   'expires_at': 1765441344,
   'status_details': None},
  {'id': 'file-1VbWfDfPRpS8aU2JQvgd53',
   'bytes': 210947051,
   'created_at': 1762849341,
   'filename': 'batch_6912eab1b75481909cbafc33a1401421_output.jsonl',
   'object': 'file',
   'purpose': 'batch_output',
   'status': 'processed',
   'expires_at': 1765441341,
   'status_details': None},
  {'id': 'file-CNWRj2LBN7XrYdttoS5BSh',
   'bytes': 8706316,
   'created_at': 1762847386,
   'filename': 'batch_for_embeddings_4.jsonl',
   'object': 'file',
   'purpose': 'batch',
   'status': 'processed',
   'expires_at': 1765439386,
   'status_details': None},
  {'id': 'file-HM7n2ZPgcYnRJmr1sEJ7DZ',
   'bytes': 9022641,
   'created_at': 1762847368,
   'filename': 'batch_for_embeddings_3.jsonl',
   'ob

In [6]:
client.batches.list().to_dict()

{'data': [{'id': 'batch_6912eab279948190b13aaccc3f5fafa9',
   'completion_window': '24h',
   'created_at': 1762847410,
   'endpoint': '/v1/embeddings',
   'input_file_id': 'file-CNWRj2LBN7XrYdttoS5BSh',
   'object': 'batch',
   'status': 'failed',
   'cancelled_at': None,
   'cancelling_at': None,
   'completed_at': None,
   'error_file_id': None,
   'errors': {'data': [{'code': 'duplicate_custom_id',
      'line': 6609,
      'message': 'The custom_id for this request is a duplicate of another request. The custom_id parameter must be unique for each request in a batch.',
      'param': 'custom_id'},
     {'code': 'duplicate_custom_id',
      'line': 6909,
      'message': 'The custom_id for this request is a duplicate of another request. The custom_id parameter must be unique for each request in a batch.',
      'param': 'custom_id'},
     {'code': 'duplicate_custom_id',
      'line': 6929,
      'message': 'The custom_id for this request is a duplicate of another request. The custom_

In [ ]:
batch_description = 

In [3]:
batch_processes = client.batches.list().to_dict()
batch_info= [
    {'batch_id': batch['id'],
     'description': batch['metadata']['description'],
    'status': batch['status'],
    'request_counts': batch['request_counts'],
    'output_file_id': batch['output_file_id']}  
            for batch in batch_processes['data'] if batch['metadata']['description'] == batch_description
    ]
batch_info

NameError: name 'batch_description' is not defined

In [ ]:
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.2f} seconds")